# ST10355869

Intel Image dataset

# Setup

In [1]:
#install the needed imports 
%pip install pyspark tensorflow pandas matplotlib seaborn scikit-learn 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pyspark.sql import SparkSession  


#import tensflow packages for model training
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dropout
from tensorflow.keras.callbacks import EarlyStopping

from pyspark.sql.functions import col, split, element_at, input_file_name, regexp_extract

#initialise spark session
spark = (
    SparkSession.builder
    .appName("PDAN8412_P3")
    .config("spark.memory.offHeap.enabled", "true")
    .config("spark.memory.offHeap.size", "10g")
    .getOrCreate()
)

# Preprocessing

deepseek version, binary file

deepseek image datasource version

In [3]:
base = "file:///C:/varsity/PostGrad/Semester 2/PDAN8412/part 3 dataset"
image_df = (
    spark.read.format("image")
    .option("recursiveFileLookup", "true")
    .load(f"{base}\\seg_train")
    .select(
        "image",
        "image.origin",
        "image.height",
        "image.width",
        "image.nChannels",
        "image.mode"
    )
    .withColumn("label", element_at(split(col("origin"), r"[/\\]"), -2))
)

image_df.printSchema()

root
 |-- image: struct (nullable = true)
 |    |-- origin: string (nullable = true)
 |    |-- height: integer (nullable = true)
 |    |-- width: integer (nullable = true)
 |    |-- nChannels: integer (nullable = true)
 |    |-- mode: integer (nullable = true)
 |    |-- data: binary (nullable = true)
 |-- origin: string (nullable = true)
 |-- height: integer (nullable = true)
 |-- width: integer (nullable = true)
 |-- nChannels: integer (nullable = true)
 |-- mode: integer (nullable = true)
 |-- label: string (nullable = true)



In [4]:
image_df.groupBy("label").count().orderBy("label").show()



+---------+-----+
|    label|count|
+---------+-----+
|buildings| 2191|
|   forest| 2271|
|  glacier| 2404|
| mountain| 2512|
|      sea| 2274|
|   street| 2382|
+---------+-----+



the classes hpw a relative balance between all classes with buildings being lowest and mountai being highest otherwise the others are about 2200-2400

checking total amount to see the split from download

In [5]:
print(f"Total number of images: {image_df.count()}")

Total number of images: 14034


checking the test set for the quantity in it

In [6]:
image_df_test = (
    spark.read.format("image")
    .option("recursiveFileLookup", "true")
    #drop invalid images that may be corrupted, these are null values
    .option("dropInvalid", True)
    .load(f"{base}\\seg_test")
    .select(
        "image",
        "image.origin",
        "image.height",
        "image.width",
        "image.nChannels",
        "image.mode"
    )
    .withColumn("label", element_at(split(col("origin"), r"[/\\]"), -2))
)
image_df_test.groupBy("label").count().orderBy("label").show()
print(f"Total number of test images: {image_df_test.count()}")

+---------+-----+
|    label|count|
+---------+-----+
|buildings|  437|
|   forest|  474|
|  glacier|  553|
| mountain|  525|
|      sea|  510|
|   street|  501|
+---------+-----+

Total number of test images: 3000


In [7]:
total_images = image_df.count() + image_df_test.count()
print(f"Total number of images: {total_images}")

Total number of images: 17034


In [8]:
test_share = image_df_test.count()/total_images
train_share = image_df.count()/total_images

print(f"Share of test images: {test_share:.2%}")
print(f"Share of train images: {train_share:.2%}")

Share of test images: 17.61%
Share of train images: 82.39%


sufficient split to use the original split, just over a 80/20 split with enough test entries in the test dataset. keeping it seperate will also allow for no data leakage to happen

# Resolution

CNN need consistent resolution for the input data so a check needs to be done to see what resolution these images are

In [9]:
from pyspark.sql import functions as F
#check the heigh and width of the images in the dataset
image_df.groupBy("height", "width").count().orderBy(F.desc("count")).show(5)


+------+-----+-----+
|height|width|count|
+------+-----+-----+
|   150|  150|13986|
|   113|  150|    7|
|   135|  150|    3|
|   111|  150|    3|
|   108|  150|    2|
+------+-----+-----+
only showing top 5 rows


the majority of the dimensions are 150x150, to see the exact number of changes the dataset will be filtered to see before and after

In [13]:
#create a copy of image dataframe to use for exploring
df = image_df

In [10]:
before_filter = image_df.count()
image_df = image_df.filter((col("height") == 150) & (col("width") == 150))
print(f"Number of images before filtering: {before_filter}")
print(f"Number of images after filtering: {image_df.count()}")

Number of images before filtering: 14034
Number of images after filtering: 13986


In [11]:
print(f"Number of images filtered out: {before_filter - image_df.count()}")

Number of images filtered out: 48


48 images were removed from the filter, to keep the size of the dataset these will still be kept since they will be able to be fit to the dimensions before training in tensorflow

In [15]:
#assign the before filtered datafrae to original dataframe
image_df = df

### Duplicates

In [18]:
df_hashed = image_df.withColumn("image_hash", F.sha2("image.data", 256))

# Group to see which hashes appear more than once
duplicate_hashes = (
    df_hashed.groupBy("image_hash")
    .agg(
        F.count("*").alias("count"),
        F.collect_list("image.origin").alias("duplicate_paths"),
    )
    .filter(F.col("count") > 1)
)
#number of duplicate images
num_duplicates = duplicate_hashes.count()
print(f"Number of duplicate images: {num_duplicates}")

Number of duplicate images: 15
